# Text Generation : AI Question–Answer System using Hugging Face Transformers

**Experiment:** Generate Text and Build a Question–Answer System
This Google Colab notebook is prepared by Dr. Sachin Balawant Takmare to generate text and make Question–Answer System:

Install the required libraries

Import the libraries

Check whether GPU is available

Load a pre-trained generative text model

Take a question from the user

Generate an answer

Display the result

Build a simple Gradio web interface

What will we build?

A simple AI Question–Answer system where the user enters a question such as:

**What is machine learning?**

and the model generates an answer.

note: This notebook uses a pre-trained model. We are not training a large language model from scratch. We are using an already trained model to generate answers.

## 1. Install Required Libraries

We use:

- **Transformers** – to load and use the pre-trained language model
- **PyTorch** – deep-learning framework used by the model
- **Accelerate** – helps load/run models efficiently
- **SentencePiece** – required by models such as FLAN-T5
- **Gradio** – creates a simple web-based interface


In [2]:
!pip -q install transformers accelerate sentencepiece gradio

## 2. Import the Required Libraries

`AutoTokenizer` converts text into tokens that the model can understand.

`AutoModelForSeq2SeqLM` loads a sequence-to-sequence text-generation model.

We use **FLAN-T5**, an instruction-following model that works well for simple question-answering demonstrations.

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gradio as gr

## 3. Check GPU Availability

Google Colab can provide an NVIDIA GPU.

The following code checks whether PyTorch can access CUDA.

If the result is:

```text
cuda
```

the model will run on the GPU.

If the result is:

```text
cpu
```

the model will run on the CPU, but generation may be slower.


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


## 4. Load the Pre-trained Text Generation Model

We use:

**`google/flan-t5-base`**

FLAN-T5 is an instruction-tuned text-to-text model. We can give it an instruction such as:

```text
Answer the following question clearly: What is artificial intelligence?
```

and it generates an answer.

### Important

The first time this cell runs, Colab downloads the model from Hugging Face. This may take some time.


In [5]:
model_id = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

model = model.to(device)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!


## 5. Create the Question–Answer Function

This function performs the main work:

**Question → Prompt → Tokenization → Model → Generated Text → Answer**

The `max_new_tokens` parameter controls approximately how long the generated answer can be.

The `num_beams` parameter can improve generation quality by considering multiple possible sequences.

In [6]:
def answer_question(question, max_new_tokens=150):
    if not question or not question.strip():
        return "Please enter a question."

    prompt = (
        "Answer the following question clearly and accurately. "
        "If the question is asking for an explanation, explain it in simple language. "
        "Question: " + question
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            num_beams=4,
            early_stopping=True
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

## 6. Test the Question–Answer System

Before creating the user interface, test the function directly in Python.

Try changing the question and run the cell again.

In [7]:
question = "What is machine learning?"

answer = answer_question(question)

print("Question:", question)
print("\nAnswer:", answer)

Question: What is machine learning?

Answer: Machine learning is a process that learns information from a computer.


## 7. Interactive Question Input

The following cell allows you to type a question directly into Colab.

Run the cell and enter your question when prompted.

In [8]:
question = input("Enter your question: ")

answer = answer_question(question)

print("\nQuestion:", question)
print("\nAnswer:", answer)

Enter your question: What is Artificial Intelligence?

Question: What is Artificial Intelligence?

Answer: Artificial Intelligence (AI) is the development of artificial intelligence.


## 8. Build a Gradio Question–Answer Interface

This follows the same idea as the **Gradio interface in the uploaded image-generation notebook**.

Instead of:

**Prompt → Image → Gallery**

our application performs:

**Question → Text Model → Answer → Textbox**

The interface contains:

- Question textbox
- Answer textbox
- Maximum answer length control
- Ask button
- Clear button

In [9]:
def qa_interface(question, max_new_tokens):
    return answer_question(
        question,
        max_new_tokens=max_new_tokens
    )

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# AI Question–Answer System")
    gr.Markdown(
        "Enter a question below and the pre-trained FLAN-T5 model will generate an answer."
    )

    with gr.Row():

        with gr.Column():

            question_box = gr.Textbox(
                label="Enter Your Question",
                placeholder="Example: What is artificial intelligence?",
                lines=4
            )

            max_tokens = gr.Slider(
                minimum=30,
                maximum=300,
                value=150,
                step=10,
                label="Maximum Answer Length"
            )

            with gr.Row():
                ask_btn = gr.Button("Ask Question")
                clear_btn = gr.ClearButton()

        with gr.Column():

            answer_box = gr.Textbox(
                label="Generated Answer",
                lines=12
            )

    ask_btn.click(
        fn=qa_interface,
        inputs=[question_box, max_tokens],
        outputs=answer_box
    )

    clear_btn.add(
        [question_box, answer_box]
    )

demo.launch()


/tmp/ipykernel_549/4292604109.py:7: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9cb10b2ba3b230583e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 9. Example Questions

After launching the application, try questions such as:

1. What is artificial intelligence?
2. What is machine learning?
3. What is deep learning?
4. What is a neural network?
5. What is the difference between AI and ML?
6. Explain supervised learning in simple words.
7. What is a dataset?
8. What is generative AI?
9. What is Python?
10. Why are GPUs useful for deep learning?

**Tip:** Try both simple and conceptual questions to observe how the generated answers change.

10. How the Complete System Works

```text
                USER
                  │
                  ▼
          Enter a Question
                  │
                  ▼
          Create an Instruction
                  │
                  ▼
             Tokenizer
                  │
                  ▼
        FLAN-T5 Pre-trained Model
                  │
                  ▼
          Generate Answer Tokens
                  │
                  ▼
             Detokenizer
                  │
                  ▼
          Display Generated Answer
                  │
                  ▼
               USER
```

### In simple words

The user asks a question.

The tokenizer converts the question into numerical tokens.

The pre-trained FLAN-T5 model processes those tokens and predicts an appropriate sequence of output tokens.

The tokenizer converts those tokens back into normal text.

Finally, Gradio displays the generated answer.


## 11. Important Parameters

| Parameter | Purpose |
|---|---|
| `model_id` | Specifies the pre-trained model |
| `max_new_tokens` | Controls maximum generated answer length |
| `num_beams` | Controls beam-search generation |
| `temperature` | Controls randomness for models/generation methods that use sampling |
| `device` | Selects GPU (`cuda`) or CPU |
| `truncation=True` | Prevents very long inputs from exceeding the model input limit |

For this notebook, `num_beams=4` is used instead of random sampling so that the demonstration remains relatively stable and easy for beginners to understand.


## 12. Limitations

This is a **generative Question–Answer system**, not a guaranteed fact-checking system.

The model may:

- Generate an incorrect answer
- Produce incomplete information
- Misunderstand an ambiguous question
- Generate confident-sounding but incorrect information

Therefore, generated answers should be verified when accuracy is important.

Also remember that the model is not automatically connected to the internet or to a private knowledge base.


## 13. Possible Improvements

Once this basic system is understood, it can be extended to:

### Level 1 – Better prompts
Give the model clearer instructions and context.

### Level 2 – Conversation
Maintain previous questions and answers to create a chatbot.

### Level 3 – Custom knowledge
Provide a document, PDF, or dataset and answer questions from that content.

### Level 4 – RAG
Use **Retrieval-Augmented Generation (RAG)**:

```text
User Question
      ↓
Retrieve Relevant Documents
      ↓
Relevant Context
      ↓
Language Model
      ↓
Answer
```

### Level 5 – Larger instruction models
Experiment with larger models when suitable GPU resources are available.


## 14. Conclusion

In this experiment, we created a simple **Generative AI Question–Answer system using Python, Hugging Face Transformers, FLAN-T5, PyTorch, and Gradio**.

The experiment demonstrates the basic workflow of a text-generation application:

**User Question → Tokenization → Pre-trained Generative Model → Generated Text → Answer**

This provides a foundation for understanding more advanced applications such as chatbots, document Question–Answering systems, and RAG-based GenAI applications.
